# Movie Recommendation System Pipeline

This notebook acts as the orchestrator to train, evaluate, and demonstrate the SequenceRecommender logic.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
from dataset import load_data, process_data, collate_fn, MovieSequenceDataset
from model import SequenceRecommender
from train import train_with_oom_fallback
from evaluate import evaluate_model
from torch.utils.data import DataLoader

plt.style.use('ggplot')

# Fix seeds globally for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)


## 1. Load Data

In [ ]:
# Load and preprocess the MovieLens data (chronological splits)
DATA_DIR = 'data'
try:
    ratings_df, movies_df = load_data(DATA_DIR)
except Exception as e:
    raise RuntimeError(f'Failed to load data from {DATA_DIR}: {e}')

# max sequence length is configurable from the Config cell above
train_data, val_data, test_data, movie2idx, user2idx = process_data(ratings_df, max_seq_len=50)

print(f'Num train pairs: {len(train_data)}')
print(f'Num val pairs: {len(val_data)}')
print(f'Num test pairs: {len(test_data)}')

# Build Dataset objects used by training/evaluation utils
train_dataset = MovieSequenceDataset(train_data)
val_dataset = MovieSequenceDataset(val_data)
test_dataset = MovieSequenceDataset(test_data)

num_users = len(user2idx)
num_movies = len(movie2idx) + 1  # reserve 0 for padding

from torch.utils.data import DataLoader
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, collate_fn=collate_fn)


In [ ]:
import os
import torch

# Cache behavior: use existing cache unless FORCE_RETRAIN is set
CACHE_FILE = 'training_cache.pt'
FORCE_RETRAIN = False
SKIP_TRAINING = os.path.exists(CACHE_FILE) and not FORCE_RETRAIN
results_arch = {}
results_opt = {}
best_model = None
best_arch = None
if SKIP_TRAINING:
    print('Found existing training cache; loading results and replaying logs.')
    cache = torch.load(CACHE_FILE, map_location=device)
    results_arch = cache.get('results_arch', {})
    results_opt = cache.get('results_opt', {})

    def _replay_epochs(name, train_losses, val_losses):
        total = max(len(train_losses), len(val_losses))
        for e in range(total):
            tr = train_losses[e] if e < len(train_losses) else float('nan')
            va = val_losses[e] if e < len(val_losses) else float('nan')
            try:
                print(f'[{name}] Epoch {e+1}/{total} - Train Loss: {tr:.4f}  Val Loss: {va:.4f}')
            except Exception:
                print(f'[{name}] Epoch {e+1}/{total} - Train Loss: {tr}  Val Loss: {va}')

    print('
Replaying saved training epochs from cache...')
    for arch, res in results_arch.items():
        _replay_epochs(f'ARCH-{arch.upper()}', res.get('train_losses', []), res.get('val_losses', []))
    for exp, res in results_opt.items():
        _replay_epochs(f'OPT-{exp}', res.get('train_losses', []), res.get('val_losses', []))


## 2. Architecture Comparison (RNN / LSTM / GRU)

In [ ]:
# Train candidate recurrent architectures unless cache is used
ARCHITECTURES = ['rnn', 'lstm', 'gru']
if not SKIP_TRAINING:
    results_arch = {}
    for arch in ARCHITECTURES:
        print(f'--- Training {arch.upper()} ---')
        create_model_fn = lambda a=arch: SequenceRecommender(num_users=num_users, num_movies=num_movies, rnn_type=a)
        model, train_losses, val_losses, final_batch_size = train_with_oom_fallback(create_model_fn, train_dataset, val_dataset, optimizer_name='adam', lr=1e-3, num_epochs=5, device=device, start_batch_size=256)
        top1, hr10 = evaluate_model(model, test_loader, device=device)
        print(f'Test Top-1: {top1:.4f}, HR@10: {hr10:.4f}')
        results_arch[arch] = {'model': model, 'train_losses': train_losses, 'val_losses': val_losses, 'top1': top1, 'hr10': hr10}
else:
    print('Skipping architecture training (cache present).')


### Architecture Comparison Results

In [ ]:
arch_summary = []
for arch, res in results_arch.items():
    arch_summary.append({'Architecture': arch.upper(), 'Top-1 Acc': res['top1'], 'HR@10': res['hr10']})
df_arch = pd.DataFrame(arch_summary) if arch_summary else pd.DataFrame([])
display(df_arch)

# Plot validation loss curves (if available)
plt.figure(figsize=(10,5))
for arch, res in results_arch.items():
    if res.get('val_losses'):
        plt.plot(res['val_losses'], label=f
)
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.title('Architecture Comparison (Validation Loss)')
plt.legend()
plt.show()

# Choose best architecture by HR@10 when available
if not df_arch.empty:
    best_arch = df_arch.sort_values(by='HR@10', ascending=False).iloc[0]['Architecture'].lower()
    print(f'Best architecture: {best_arch.upper()}')
else:
    print('No architecture results available yet.')


## 3. Optimizer & Learning Rate Tuning

In [ ]:
# Grid search over optimizers and learning rates using the chosen backbone
OPTIMIZERS = ['sgd', 'adam']
LEARNING_RATES = [1e-2, 1e-3, 1e-4]
results_opt = {}
if not SKIP_TRAINING:
    if best_arch is None:
        # If architecture selection was skipped (e.g., using cache), pick a default
        default_backbone = 'lstm'
    else:
        default_backbone = best_arch
    print(f'Running optimizer grid search on backbone: {default_backbone.upper()}')
    for opt in OPTIMIZERS:
        for lr in LEARNING_RATES:
            exp_name = f

            print(f'--- Experiment: {exp_name} ---')
            create_model_fn = lambda o=opt: SequenceRecommender(num_users=num_users, num_movies=num_movies, rnn_type=default_backbone)
            model, train_losses, val_losses, _ = train_with_oom_fallback(create_model_fn, train_dataset, val_dataset, optimizer_name=opt, lr=lr, num_epochs=5, device=device, start_batch_size=256)
            top1, hr10 = evaluate_model(model, test_loader, device=device)
            results_opt[exp_name] = {'model': model, 'train_losses': train_losses, 'val_losses': val_losses, 'top1': top1, 'hr10': hr10, 'opt': opt, 'lr': lr}
else:
    print('Skipping optimizer grid search (cache present).')


### Optimizer Comparison Results and Save

Below we summarize optimizer experiments and save the training cache for future runs.

In [ ]:
opt_summary = []
for exp_name, res in results_opt.items():
    opt_summary.append({'Experiment': exp_name, 'Optimizer': res['opt'].upper(), 'Learning Rate': res['lr'], 'Top-1 Acc': res['top1'], 'HR@10': res['hr10']})
df_opt = pd.DataFrame(opt_summary) if opt_summary else pd.DataFrame([])
display(df_opt)

plt.figure(figsize=(10,5))
for exp_name, res in results_opt.items():
    if res.get('val_losses'):
        plt.plot(res['val_losses'], label=f
)
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.title('Optimizer & LR Tuning (Validation Loss)')
plt.legend()
plt.show()

# Select best experiment by HR@10 when available
if not df_opt.empty:
    best_exp_row = df_opt.sort_values(by='HR@10', ascending=False).iloc[0]
    best_exp_name = best_exp_row['Experiment']
    best_model = results_opt[best_exp_name]['model']
    print(f'Winning experiment: {best_exp_name}')
else:
    print('No optimizer results available to select best experiment.')

# Save cache once if we ran training
if not SKIP_TRAINING:
    print('Saving training cache to disk for faster future runs...')
    torch.save({'results_arch': results_arch, 'results_opt': results_opt}, CACHE_FILE)


## 4. Interactive Recommendation Demo

In [ ]:
import re

def normalize_title(title):
    if not isinstance(title, str):
        return ''
    title = title.lower().strip()
    title = re.sub(r'\(\d{4}\)', '', title).strip()
    return title

# Prepare mapping dictionaries used by inference
movies_df['normalized_title'] = movies_df['title'].astype(str).apply(normalize_title)
idx2movie = {idx: mid for mid, idx in movie2idx.items()}
mid2title = dict(zip(movies_df['movie_id'], movies_df['title']))

def _find_movie_id_by_title(query):
    q = normalize_title(query)
    if not q:
        return None
    # Try exact match first, then contains
    exact = movies_df[movies_df['normalized_title'] == q]
    if not exact.empty:
        return exact.iloc[0]['movie_id']
    contains = movies_df[movies_df['normalized_title'].str.contains(q, na=False)]
    if not contains.empty:
        return contains.iloc[0]['movie_id']
    return None

def predict_next_movie(input_titles, model_to_use):
    model_to_use.eval()
    movie_indices = []
    matched_titles = []
    for t in input_titles:
        mid = _find_movie_id_by_title(t)
        if mid is None:
            print(f'Could not find match for: {t}')
            continue
        if mid not in movie2idx:
            print(f'Movie id {mid} not in training split mapping; skipping')
            continue
        movie_indices.append(movie2idx[mid])
        matched_titles.append(mid2title.get(mid, str(mid)))
    if not movie_indices:
        print('No valid movies found to build a recommendation sequence.')
        return None
    print(f'Mapped input sequence: {matched_titles}')
    seq_tensor = torch.tensor([movie_indices], dtype=torch.long).to(device)
    user_tensor = torch.tensor([0], dtype=torch.long).to(device)
    lengths_tensor = torch.tensor([len(movie_indices)], dtype=torch.long).to(device)
    with torch.no_grad():
        probs = model_to_use.predict(user_tensor, seq_tensor, lengths_tensor)
    probs[0, 0] = -1.0
    pred_idx = int(probs.argmax(dim=-1).item())
    pred_mid = idx2movie.get(pred_idx)
    pred_title = mid2title.get(pred_mid, 'Unknown') if pred_mid is not None else 'Unknown'
    print(f'RECOMMENDED NEXT: {pred_title}')
    return pred_mid

def predict_next_movie_simple(history):
    # Non-widget fallback: uses best_model if available
    if 'best_model' in globals() and best_model is not None:
        return predict_next_movie(history, best_model)
    else:
        print('Best model is not available for inference. Ensure training or cache load completed.')
        return None


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

all_titles = movies_df['title'].dropna().unique().tolist()
movie_input = widgets.Combobox(placeholder='Type a movie title (e.g., Toy Story)', options=all_titles, description='Movie:', ensure_option=False, layout=widgets.Layout(width='400px'))
add_button = widgets.Button(description='Add to history', button_style='info', icon='plus')
predict_button = widgets.Button(description='Predict next', button_style='success', icon='magic')
clear_button = widgets.Button(description='Clear', button_style='warning', icon='refresh')
history_display = widgets.Output()
result_display = widgets.Output()
current_history = []
def add_movie(b):
    if movie_input.value:
        current_history.append(movie_input.value)
        with history_display:
            clear_output()
            print(f'History: {', '.join(current_history)}')
        movie_input.value = ''
def clear_history(b):
    current_history.clear()
    movie_input.value = ''
    with history_display:
        clear_output()
        print('History: (empty)')
    with result_display:
        clear_output()
def run_prediction(b):
    with result_display:
        clear_output()
        if not current_history:
            print('Please add at least one movie to history before predicting!')
            return
        print(f'Predicting based on {len(current_history)} movies...')
        predict_next_movie(current_history, best_model if 'best_model' in globals() else None)
add_button.on_click(add_movie)
clear_button.on_click(clear_history)
predict_button.on_click(run_prediction)
with history_display:
    print('History: (empty)')
buttons_layout = widgets.HBox([add_button, clear_button, predict_button])
main_ui = widgets.VBox([widgets.HTML('<h2>Interactive Recommendation Demo</h2>'), widgets.HTML('<p>Enter movie titles and press <b>Add to history</b>. Then press <b>Predict next</b>.</p>'), movie_input, buttons_layout, history_display, result_display])
display(main_ui)
